# 04 Gold

Publica el día del job y cualquier día anterior cuyo `batch_alta` en silver sea
más nuevo que el ya escrito en `oceanwatch_control_oro`. `replaceWhere` sustituye
solo la partición de esa fecha, así repetir la tarea no duplica.

- **Trayectorias:** una fila por buque, con la distancia del día (haversine, radio 6371 km,
  la misma fórmula de la Parte 1) entre posiciones consecutivas.
- **Presencia en puerto:** el buque tiene al menos una posición en la celda H3 resolución 7
  del puerto o en el anillo vecino (k = 1). La Parte 1 usó resolución 8 exacta y no cruzó
  con ningún puerto, esta vez la celda es más amplia.
- **Tráfico:** posiciones y buques por celda H3 resolución 8 y hora UTC.


In [ ]:
# Capturar los parámetros del notebook llamador antes de %run.
dbutils.widgets.text("fecha", "2023-06-08")
dbutils.widgets.text("ancla_replay", "")
dbutils.widgets.text("inicio_run", "")
ow_argumentos = tuple(dbutils.widgets.get(k).strip() for k in ("fecha", "ancla_replay", "inicio_run"))


In [ ]:
%run ./_config


## Trayectorias


In [ ]:
from pyspark.sql import Window
from pyspark.sql import functions as F

def trayectorias(posiciones):
    orden = Window.partitionBy("fecha", "MMSI").orderBy("BaseDateTime")
    orden_fin = Window.partitionBy("fecha", "MMSI").orderBy(F.desc("BaseDateTime"))
    lat1 = F.radians(F.lag("LAT").over(orden))
    lon1 = F.radians(F.lag("LON").over(orden))
    lat2 = F.radians("LAT")
    lon2 = F.radians("LON")
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = F.sin(dlat / 2) ** 2 + F.cos(lat1) * F.cos(lat2) * F.sin(dlon / 2) ** 2
    tramo = F.when(
        F.lag("LAT").over(orden).isNotNull(),
        2 * F.lit(6371.0) * F.asin(F.sqrt(F.least(F.lit(1.0), F.greatest(F.lit(0.0), a)))),
    ).otherwise(F.lit(0.0))
    marcado = (
        posiciones.withColumn("tramo_km", tramo)
        .withColumn("lat_inicio", F.first("LAT").over(orden))
        .withColumn("lon_inicio", F.first("LON").over(orden))
        .withColumn("lat_fin", F.first("LAT").over(orden_fin))
        .withColumn("lon_fin", F.first("LON").over(orden_fin))
        .withColumn("inicio", F.first("BaseDateTime").over(orden))
        .withColumn("fin", F.first("BaseDateTime").over(orden_fin))
    )
    return marcado.groupBy("fecha", "MMSI").agg(
        F.count("*").alias("n_posiciones"),
        F.max("inicio").alias("inicio"),
        F.max("fin").alias("fin"),
        F.max("lat_inicio").alias("lat_inicio"),
        F.max("lon_inicio").alias("lon_inicio"),
        F.max("lat_fin").alias("lat_fin"),
        F.max("lon_fin").alias("lon_fin"),
        F.sum("tramo_km").alias("distancia_km"),
        F.avg("SOG").alias("sog_media"),
    )


def publicar(df, tabla: str, iso: str) -> None:
    (
        df.write.format("delta")
        .mode("overwrite")
        .option("replaceWhere", f"fecha = '{iso}'")
        .saveAsTable(tabla)
    )
    print("Publicado", tabla)


## Presencia en puerto


In [ ]:
puertos = (
    spark.table(PUERTOS)
    .where(F.col("latitud").isNotNull() & F.col("longitud").isNotNull())
    .withColumn(
        "h3_res7",
        F.explode(F.expr(
            "transform(h3_kring(h3_longlatash3(longitud, latitud, 7), 1), c -> h3_h3tostring(c))"
        )),
    )
    .select("nombre", "pais", "h3_res7").dropDuplicates(["nombre", "pais", "h3_res7"])
)


def presencia_diaria(dia, puertos):
    posiciones = dia.withColumn("h3_res7", F.expr("h3_longlatash3string(LON, LAT, 7)"))
    return (
        posiciones.join(puertos, "h3_res7", "inner")
        .groupBy("fecha", "MMSI", "nombre", "pais")
        .agg(
            F.count("*").alias("n_posiciones"),
            F.min("BaseDateTime").alias("inicio"),
            F.max("BaseDateTime").alias("fin"),
        )
        .withColumnRenamed("nombre", "puerto")
    )


## Tráfico por celda y hora


In [ ]:
def trafico_diario(dia):
    return (
        dia.withColumn("hora", F.hour("BaseDateTime"))
        .withColumn("h3_cell", F.expr("h3_longlatash3string(LON, LAT, 8)"))
        .groupBy("fecha", "hora", "h3_cell")
        .agg(
            F.count("*").alias("n_posiciones"),
            F.countDistinct("MMSI").alias("n_buques"),
        )
    )


## Publicación diaria y rezagos

El control por batch detecta fechas anteriores afectadas por llegadas tardías. Se confirma una fecha después de las tres escrituras. Repetir una fecha sustituye sus particiones.


In [ ]:
corte = fecha_dia.isoformat()
cambios = spark.sql(f"""WITH s AS (
  SELECT fecha_dia AS fecha, max(batch_alta) AS batch FROM {SILVER}
  WHERE fecha_dia <= DATE '{corte}' GROUP BY fecha_dia)
SELECT s.fecha, s.batch FROM s LEFT JOIN {CONTROL_ORO} c ON c.fecha=s.fecha
WHERE c.fecha IS NULL OR s.batch>c.batch_plata OR s.fecha=DATE '{corte}' ORDER BY s.fecha""").collect()
for cambio in cambios:
    iso = cambio.fecha.isoformat()
    dia = (spark.table(SILVER).where(F.col("fecha_dia") == F.lit(cambio.fecha))
           .withColumn("fecha", F.col("fecha_dia")))
    publicar(trayectorias(dia), GOLD_TRAY, iso)
    publicar(presencia_diaria(dia, puertos), GOLD_PUERTO, iso)
    publicar(trafico_diario(dia), GOLD_TRAFICO, iso)
    spark.sql(f"""MERGE INTO {CONTROL_ORO} d USING
        (SELECT DATE '{iso}' fecha, {int(cambio.batch)}L batch_plata, current_timestamp() publicado_en) s
        ON d.fecha=s.fecha WHEN MATCHED THEN UPDATE SET * WHEN NOT MATCHED THEN INSERT *""")
if not any(x.fecha == fecha_dia for x in cambios):
    raise ValueError(f"El día {corte} carece de posiciones válidas. Revisa landing y las métricas de Silver.")
